# Chantier 2 — Precompute backtest 2025

Genere `data/processed/backtest_2025.parquet` avec les colonnes :
`(date_cmd, semaine_iso, code_client_freq, code_article_freq, qte_reelle,
  pred_xgb_v2, pred_lgbm_p50, pred_catboost, pred_stacking, pred_baseline, pred_ddmrp)`

Permet a la page Backtest Streamlit de filtrer/aggreger sans recalculer.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import joblib

from dashboard.utils.features import FEATURES_V2, TARGET_QTE
DATA = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'

In [ ]:
df_train = pd.read_parquet(DATA / 'split_train_v3_features.parquet')
df_test = pd.read_parquet(DATA / 'split_test_v3_features.parquet')
X_test = df_test[FEATURES_V2]

# Baseline = mediane historique par couple
couple_median = df_train.groupby(['code_client_freq', 'code_article_freq'])[TARGET_QTE].median()
global_median = df_train[TARGET_QTE].median()
baseline_pred = df_test.set_index(['code_client_freq', 'code_article_freq']).index.map(
    lambda k: couple_median.get(k, global_median)
).astype(float)
print('Baseline OK')

In [ ]:
xgb_v2 = joblib.load(MODELS / 'xgboost_optuna_v2.pkl')
lgbm_p50 = joblib.load(MODELS / 'lgbm_quantile_p50.pkl')
preds = {
    'pred_xgb_v2': np.clip(np.expm1(xgb_v2.predict(X_test)), 0, None),
    'pred_lgbm_p50': np.clip(np.expm1(lgbm_p50.predict(X_test)), 0, None),
    'pred_baseline': baseline_pred.values,
}
for k, v in preds.items():
    print(k, 'min=%.2f max=%.2f mean=%.2f' % (v.min(), v.max(), v.mean()))

In [ ]:
# CatBoost + Stacking si dispo
cat_path = MODELS / 'catboost_v2.pkl'
stack_path = MODELS / 'stacking_ridge.pkl'
if cat_path.exists():
    cat = joblib.load(cat_path)
    preds['pred_catboost'] = np.clip(np.expm1(cat.predict(X_test)), 0, None)
else:
    preds['pred_catboost'] = np.nan
    print('CatBoost absent, colonne en NaN')

if stack_path.exists():
    ridge = joblib.load(stack_path)
    comps = json.load(open(MODELS / 'stacking_components.json'))['components']
    P = np.column_stack([
        preds['pred_xgb_v2'] if c == 'xgb_v2' else
        preds['pred_lgbm_p50'] if c == 'lgbm_p50' else
        preds['pred_catboost'] if c == 'catboost' else
        np.zeros(len(df_test))
        for c in comps
    ])
    preds['pred_stacking'] = np.clip(ridge.predict(P), 0, None)
else:
    preds['pred_stacking'] = np.nan
    print('Stacking absent')

In [ ]:
# DDMRP : on reutilise comparison_ia_vs_ddmrp si genere
ddmrp_path = DATA / 'comparison_ia_vs_ddmrp.parquet'
df_test['semaine_iso'] = df_test['date_cmd'].dt.isocalendar().week.astype(int)
df_test['annee_iso'] = df_test['date_cmd'].dt.isocalendar().year.astype(int)

backtest = pd.DataFrame({
    'date_cmd': df_test['date_cmd'].values,
    'annee_iso': df_test['annee_iso'].values,
    'semaine_iso': df_test['semaine_iso'].values,
    'code_client_freq': df_test['code_client_freq'].values,
    'code_article_freq': df_test['code_article_freq'].values,
    'qte_reelle': df_test[TARGET_QTE].values,
    **preds,
})
backtest.to_parquet(DATA / 'backtest_2025.parquet')
print('Backtest precompute :', backtest.shape)
print(backtest.head())

## Verification : metriques globales par modele

In [ ]:
from sklearn.metrics import mean_absolute_error
for col in ['pred_xgb_v2', 'pred_lgbm_p50', 'pred_catboost', 'pred_stacking', 'pred_baseline']:
    if backtest[col].notna().all():
        mae = mean_absolute_error(backtest['qte_reelle'], backtest[col])
        wape = np.sum(np.abs(backtest['qte_reelle']-backtest[col])) / np.sum(np.abs(backtest['qte_reelle']))
        print(f'{col}: MAE={mae:.3f}  WAPE={wape:.3f}')
    else:
        print(f'{col}: SKIP (NaN)')